In [0]:
!pip install kaggle

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

os.environ["KAGGLE_USERNAME"] = "dnandi"
os.environ["KAGGLE_KEY"] = "KGAT_24e60868ea7a8fc8dc48dba5439a3692"

print("Kaggle credentials configured!")

Kaggle credentials configured!


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

DataFrame[]

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors


100%|██████████| 4.29G/4.29G [00:34<00:00, 135MB/s]


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: 2019-Nov.csv            
  inflating: 2019-Oct.csv            
total 18G
-rwxrwxrwx 1 spark-268ce80d-d215-4805-867d-4b nogroup 8.4G Feb 22 15:43 2019-Nov.csv
-rwxrwxrwx 1 spark-268ce80d-d215-4805-867d-4b nogroup 5.3G Feb 22 15:45 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 15:36 delta
-rwxrwxrwx 1 spark-268ce80d-d215-4805-867d-4b nogroup 4.3G Feb 22 15:42 ecommerce-behavior-data-from-multi-category-store.zip
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 15:36 oct_delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 15:36 outputs


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

total 14G
-rwxrwxrwx 1 spark-268ce80d-d215-4805-867d-4b nogroup 8.4G Feb 22 15:43 2019-Nov.csv
-rwxrwxrwx 1 spark-268ce80d-d215-4805-867d-4b nogroup 5.3G Feb 22 15:45 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 15:36 delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 15:36 oct_delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Feb 22 15:36 outputs


In [0]:
%restart_python

In [0]:
df_n = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")

In [0]:
df = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")

In [0]:
print(f"October 2019 - Total Events: {df.count():,}")
print("\n" + "="*60)
print("SCHEMA:")
print("="*60)
df.printSchema()

October 2019 - Total Events: 42,448,765

SCHEMA:
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)



In [0]:
print("\n" + "="*60)
print("SAMPLE DATA (First 5 rows):")
print("="*60)
df.show(5, truncate=False)


SAMPLE DATA (First 5 rows):
+-----------------------+----------+----------+-------------------+-----------------------------------+--------+------+---------+------------------------------------+
|_c0                    |_c1       |_c2       |_c3                |_c4                                |_c5     |_c6   |_c7      |_c8                                 |
+-----------------------+----------+----------+-------------------+-----------------------------------+--------+------+---------+------------------------------------+
|event_time             |event_type|product_id|category_id        |category_code                      |brand   |price |user_id  |user_session                        |
|2019-10-01 00:00:00 UTC|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79 |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-10-01 00:00:00 UTC|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.20 |554748717|9333dfb

In [0]:
print("Current columns:", df.columns)
column_map = {
    "_c0": "event_time", "_c1": "event_type", "_c2": "product_id", 
    "_c3": "category_id", "_c4": "category_code", "_c5": "brand", 
    "_c6": "price", "_c7": "user_id", "_c8": "user_session"
}
for old_name, new_name in column_map.items():
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name, new_name)

print("✅ Columns renamed!")
df.printSchema()
df.show(2)

Current columns: ['_c0', '_c1', '_c2', '_c3', '_c4', '_c5', '_c6', '_c7', '_c8']
✅ Columns renamed!
root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_session: string (nullable = true)

+--------------------+----------+----------+-------------------+-------------+--------+-----+---------+--------------------+
|          event_time|event_type|product_id|        category_id|category_code|   brand|price|  user_id|        user_session|
+--------------------+----------+----------+-------------------+-------------+--------+-----+---------+--------------------+
|          event_time|event_type|product_id|        category_id|category_code|   brand|price|  user_id|        user_session|
|2019-10-01 00:00:...| 

In [0]:
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/oct_delta"
df.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .save(delta_path)
print(f"Row count: {spark.read.format('delta').load(delta_path).count()}")
df.printSchema()

Row count: 42448765
root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_session: string (nullable = true)



In [0]:
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/oct_delta"
print("Delta table ready at:", delta_path)
print(f"Initial row count: {spark.read.format('delta').load(delta_path).count()}")
spark.sql(f"DESCRIBE DETAIL delta.`{delta_path}`").select("numFiles", "sizeInBytes", "properties").show(truncate=False)

Delta table ready at: /Volumes/workspace/ecommerce/ecommerce_data/oct_delta
Initial row count: 42448765
+--------+-----------+-------------------------------------+
|numFiles|sizeInBytes|properties                           |
+--------+-----------+-------------------------------------+
|43      |1493399323 |{delta.enableDeletionVectors -> true}|
+--------+-----------+-------------------------------------+



In [0]:
dfn = spark.read.option("inferSchema", "true").csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv", header=False)
dfn = (dfn.withColumnRenamed("_c0", "event_time")
         .withColumnRenamed("_c1", "event_type")
         .withColumnRenamed("_c2", "product_id")
         .withColumnRenamed("_c3", "category_id")
         .withColumnRenamed("_c4", "category_code")
         .withColumnRenamed("_c5", "brand")
         .withColumnRenamed("_c6", "price")
         .withColumnRenamed("_c7", "user_id")
         .withColumnRenamed("_c8", "user_session"))
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/oct_delta"
print("📈 Append 1: Full November data")
dfn.write.format("delta").mode("append").save(delta_path)
print("📈 Append 2: 1M row subset")
dfn.limit(1000000).write.format("delta").mode("append").save(delta_path)
print("📈 Append 3: 500K row subset")
dfn.limit(500000).write.format("delta").mode("append").save(delta_path)
print(f"✅ Total rows after 3 appends: {spark.read.format('delta').load(delta_path).count()}")

📈 Append 1: Full November data
📈 Append 2: 1M row subset
📈 Append 3: 500K row subset
✅ Total rows after 3 appends: 111450745


In [0]:
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/oct_delta"
print("📊 BEFORE OPTIMIZE - File fragmentation:")
spark.sql(f"DESCRIBE DETAIL delta.`{delta_path}`").select("numFiles", "sizeInBytes").show(truncate=False)

📊 BEFORE OPTIMIZE - File fragmentation:
+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|113     |4133657837 |
+--------+-----------+



In [0]:
print("🔧 Running OPTIMIZE (file compaction only)...")
spark.sql(f"OPTIMIZE delta.`{delta_path}`")

print("\n📊 BEFORE OPTIMIZE:")
spark.sql(f"DESCRIBE DETAIL delta.`{delta_path}`").select("numFiles", "sizeInBytes").show(truncate=False)

print("\n📊 AFTER OPTIMIZE:")
spark.sql(f"DESCRIBE DETAIL delta.`{delta_path}`").select("numFiles", "sizeInBytes").show(truncate=False)

print("\n✅ OPTIMIZE complete!")
print("• Small files from appends compacted into optimal sizes ✓")
print("• Expect fewer files, larger average size after compaction")
print("• Query performance improves due to fewer file reads")

🔧 Running OPTIMIZE (file compaction only)...

📊 BEFORE OPTIMIZE:
+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|17      |4496237517 |
+--------+-----------+


📊 AFTER OPTIMIZE:
+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|17      |4496237517 |
+--------+-----------+


✅ OPTIMIZE complete!
• Small files from appends compacted into optimal sizes ✓
• Expect fewer files, larger average size after compaction
• Query performance improves due to fewer file reads


In [0]:
print("⏱️  Query performance test (user filter):")
%time
result = spark.read.format("delta").load(delta_path)\
  .filter("user_id = '54131214072d76fde-8bb3-4e00-8c23-a032dfed738c'")\
  .count()
print(f"✅ User events found: {result}")

⏱️  Query performance test (user filter):
CPU times: user 3 μs, sys: 1 μs, total: 4 μs
Wall time: 7.39 μs
✅ User events found: 0
